# Load data

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
%matplotlib qt

### functions

In [2]:
def compute_pos(step_size, image_size):
    # Generate positions for updating symmetry image
    positions = []
    for j in range(image_size):
        for i in range(0, image_size, step_size):
            positions.append(np.array([i, j]))
    return np.array(positions)

# def convert_ij_to_pq(i, j, kernel_size):
#     # Convert (i, j) to (p, q) with kernel size
#     p = i + (kernel_size - 1) / 2
#     q = j + (kernel_size - 1) / 2
#     return [int(p), int(q)]

def update_symmetry(animated_image, image, i, j, step_size, image_size):
    # Update part of the animated image based on the original image
    updated_i = np.min([i + step_size, image_size])
    animated_image[j][i:updated_i] = image[j][i:updated_i]
    return animated_image

def update_kernal(display_image, p, q, kernel_size, kernal_thickness):
    # Draw the sliding kernel onto the display image
    display_image[q:q + kernal_thickness, p:p + kernel_size] = 255
    display_image[q + kernel_size - kernal_thickness:q + kernel_size, p:p + kernel_size] = 255
    display_image[q:q + kernel_size, p:p + kernal_thickness] = 255
    display_image[q:q + kernel_size, p + kernel_size - kernal_thickness:p + kernel_size] = 255
    return display_image

def normalize_imgs_to_255(img):
    img-=img.min()
    img/=img.max()
    img = img*255
    return img

## load an image

In [3]:
img = np.load('.\\data\\STEM img.npy')
mirror = np.load('.\\data\\mirror img.npy')

### parameters setting

In [11]:
image_size = mirror.shape[0]
step_size = 120
kernel_size = 27
kernal_thickness = 3

In [12]:
kernel_border = int((kernel_size-1) / 2)

In [13]:
mirror = mirror[kernel_border:image_size-kernel_border, kernel_border:image_size-kernel_border]
image_size = mirror.shape[0]

In [14]:
mirror = normalize_imgs_to_255(mirror)
img = normalize_imgs_to_255(img)

In [24]:
# animated_image1 = np.full_like(mirror, 255)
animated_image1 = np.ma.array(mirror.copy(),mask = np.ones(mirror.shape))
white_img = np.full_like(mirror, 255)
display_image = img.copy()

# Create the figure and subplots
fig = plt.figure(figsize=(15, 6), dpi=80)
gs = GridSpec(1, 3)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[2])
ax3 = fig.add_subplot(gs[1])

# Adjust layout
ax1.axis("off")
ax2.axis("off")
ax3.axis("off")

pos1 = ax1.get_position()  
pos2 = ax3.get_position()   

ax1.set_position([pos1.x0, pos1.y0, pos1.width, pos1.height])  
ax3.set_position([pos1.x0 + 2 * (pos2.x0 - pos1.x0), pos2.y0, pos2.width, pos2.height])


# Left image (sliding kernel visualization)
# im1 = ax1.imshow(display_image,cmap = 'gray', vmin=0, vmax=255)
im1 = ax1.imshow(display_image, vmin=0, vmax=255)
# Right image (gradually revealed pixels)
# im_white = ax2.imshow(white_img, cmap = 'gray',vmin=0, vmax=255)
# im2 = ax3.imshow(animated_image1, cmap = 'gray',vmin=0, vmax=255)
im_white = ax2.imshow(white_img, vmin=0, vmax=255)
im2 = ax3.imshow(animated_image1, vmin=0, vmax=255)

# Compute positions for the sliding kernel
positions = compute_pos(step_size, image_size)

def update(frame):
    global frame_idx, display_image, animated_image1
    i, j = positions[frame]
    display_image = img.copy()
    animated_image1 = update_symmetry(animated_image1, mirror, i, j, step_size, image_size)
    display_image = update_kernal(display_image, i, j, kernel_size, kernal_thickness)
    
    im1.set_array(display_image)
    im2.set_array(animated_image1)
    return [im1, im2]

# Create the animation
ani = FuncAnimation(
    fig,
    update,
    # frames=len(positions) - 1,
    frames = 50,
    interval=1,  # Interval between frames in milliseconds
    blit=True,
)

# Save the animation as a GIF
ani.save(
    "reflectional_symmetry_animation1.gif",
    writer=PillowWriter(fps=800),
    savefig_kwargs={"transparent": True, "pad_inches": 0},
)

plt.close(fig)

In [ ]:
plt.imshow(img)

In [20]:
masked_img = img.copy()
masked_img = np.ma.array(masked_img,mask = np.ones(img.shape))

In [22]:
plt.imshow(img)

In [17]:
a = np.array([1,2,3,4,5])
b = np.ma.array(a, mask = [0,0,1,0,0])

In [19]:
b[2] = 5
b

masked_array(data=[1, 2, 5, 4, 5],
             mask=[False, False, False, False, False],
       fill_value=999999)

In [18]:
b

masked_array(data=[1, 2, --, 4, 5],
             mask=[False, False,  True, False, False],
       fill_value=999999)